# STUDD trên Insects

**Khoa · Kaggle CPU · BL-st và STUDD · Seed 42**

- Teacher **XGB**, student **XGB** theo Bảng 2.
- Dùng **5,325** mẫu theo thứ tự nguồn. **500** mẫu đầu để huấn luyện, **4,825** mẫu tiếp để đánh giá.
- Kappa trong Bảng 3 là **0.81** cho BL-st và **0.81** cho STUDD.

Bật **Internet** và để **Accelerator = None**. Chạy từng cell từ trên xuống. Cell cuối gửi hai run khi đặt `LOG_TO_WANDB = True` và cấp secret `WANDB_API_KEY`. 

Cặp mô hình, W và delta theo [paper STUDD](https://doi.org/10.1007/s10994-022-06188-7). Detector dùng lớp Page–Hinkley từ scikit-multiflow 0.5.3 với delta=0.001. Seed và tham số mô hình nằm trong cell 4. Nhóm dùng trực tiếp cặp mô hình trong paper. Phần đối chiếu với mã tác giả cần một lượt chạy riêng.

In [ ]:
# Cell 1 — Cài thư viện trước khi import NumPy.
%pip install --only-binary=:all: "wandb==0.28.1" "xgboost==3.1.3"

In [ ]:
# Cell 2 — Cấu hình
DATASET = 'Insects'
SLUG = 'insects'
DATA_URL = 'https://huggingface.co/datasets/pham121/studd-insects/resolve/4fa53b650844ef482a59e486a2927e84cf64e09a/Insects.data'
EXPECTED_SHA256 = '5c588472b0e8f4ab4149754192d14a5514d27001a2e55cd64c39f41f2e0c374d'
ARCHIVE_MEMBER = ''
SOURCE_FORMAT = 'raw'
EXPECTED_SHAPE = (5325, 50)
EXPECTED_CLASSES = 5
W = 500
TEACHER = 'XGB'
STUDENT = 'XGB'
PAPER_KAPPA = {'BL-st': 0.81, 'STUDD': 0.81}
SEED = 42
LOG_TO_WANDB = False
DETECTOR_SOURCE = "scikit-multiflow-0.5.3-page-hinkley"
DETECTOR_SOURCE_URL = 'https://github.com/scikit-multiflow/scikit-multiflow/blob/0.5.3/src/skmultiflow/drift_detection/page_hinkley.py'

# Page–Hinkley từ scikit-multiflow 0.5.3.
# Adapter chỉ cung cấp API của lớp cơ sở mà PageHinkley sử dụng.
class BaseDriftDetector:
    def reset(self):
        self.in_concept_change = False
        self.in_warning_zone = False
        self.estimation = 0.0
        self.delay = 0

    def detected_change(self):
        return self.in_concept_change


class PageHinkley(BaseDriftDetector):
    """ Page-Hinkley method for concept drift detection.

    Notes
    -----
    This change detection method works by computing the observed 
    values and their mean up to the current moment. Page-Hinkley
    won't output warning zone warnings, only change detections. 
    The method works by means of the Page-Hinkley test [1]_. In general
    lines it will detect a concept drift if the observed mean at 
    some instant is greater then a threshold value lambda.

    References
    ----------
    .. [1] E. S. Page. 1954. Continuous Inspection Schemes.
       Biometrika 41, 1/2 (1954), 100–115.
    
    Parameters
    ----------
    min_instances: int (default=30)
        The minimum number of instances before detecting change.
    delta: float (default=0.005)
        The delta factor for the Page Hinkley test.
    threshold: int (default=50)
        The change detection threshold (lambda).
    alpha: float (default=1 - 0.0001)
        The forgetting factor, used to weight the observed value 
        and the mean.
    
    Examples
    --------
    >>> # Imports
    >>> import numpy as np
    >>> from skmultiflow.drift_detection import PageHinkley
    >>> ph = PageHinkley()
    >>> # Simulating a data stream as a normal distribution of 1's and 0's
    >>> data_stream = np.random.randint(2, size=2000)
    >>> # Changing the data concept from index 999 to 2000
    >>> for i in range(999, 2000):
    ...     data_stream[i] = np.random.randint(4, high=8)
    >>> # Adding stream elements to the PageHinkley drift detector and verifying if drift occurred
    >>> for i in range(2000):
    ...     ph.add_element(data_stream[i])
    ...     if ph.detected_change():
    ...         print('Change has been detected in data: ' + str(data_stream[i]) + ' - of index: ' + str(i))
    
    """
    def __init__(self, min_instances=30, delta=0.005, threshold=50, alpha=1 - 0.0001):
        super().__init__()
        self.min_instances = min_instances
        self.delta = delta
        self.threshold = threshold
        self.alpha = alpha
        self.x_mean = None
        self.sample_count = None
        self.sum = None
        self.reset()

    def reset(self):
        """ reset

        Resets the change detector parameters.

        """
        super().reset()
        self.sample_count = 1
        self.x_mean = 0.0
        self.sum = 0.0

    def add_element(self, x):
        """ Add a new element to the statistics
        
        Parameters
        ----------
        x: numeric value
            The observed value, from which we want to detect the
            concept change.
        
        Notes
        -----
        After calling this method, to verify if change was detected, one 
        should call the super method detected_change, which returns True 
        if concept drift was detected and False otherwise.
        
        """
        if self.in_concept_change:
            self.reset()

        self.x_mean = self.x_mean + (x - self.x_mean) / float(self.sample_count)
        self.sum = max(0., self.alpha * self.sum + (x - self.x_mean - self.delta))

        self.sample_count += 1

        self.estimation = self.x_mean
        self.in_concept_change = False
        self.in_warning_zone = False

        self.delay = 0

        if self.sample_count < self.min_instances:
            return None

        if self.sum > self.threshold:
            self.in_concept_change = True

In [ ]:
# Cell 3 — Tải dữ liệu và kiểm tra
from hashlib import sha256
from importlib.metadata import version
from io import BytesIO
from urllib.parse import urlparse
from zipfile import ZipFile

import numpy as np
import pandas as pd
import requests

if not DATA_URL or urlparse(DATA_URL).scheme != "https" or urlparse(DATA_URL).hostname != "huggingface.co":
    raise ValueError("Điền URL tải tệp Hugging Face ở cell 2 theo hướng dẫn đầu notebook.")

response = requests.get(DATA_URL, timeout=120)
response.raise_for_status()
content = response.content
if sha256(content).hexdigest() != EXPECTED_SHA256:
    raise ValueError("SHA-256 không khớp. Kiểm tra URL và phiên bản dữ liệu.")

if ARCHIVE_MEMBER:
    with ZipFile(BytesIO(content)) as archive:
        content = archive.read(ARCHIVE_MEMBER)

if SOURCE_FORMAT == "at_data":
    with BytesIO(content) as stream:
        if stream.readline().strip().lower() != b"@data":
            raise ValueError("Thiếu dòng @data của Arabic.")
        data = pd.read_csv(stream, header=None)
elif SOURCE_FORMAT == "sex_csv":
    data = pd.read_csv(BytesIO(content))
    if list(data.columns[-2:]) != ["sex", "digit"] or set(data["sex"]) != {"male", "female"}:
        raise ValueError("Sai cấu trúc cột sex/digit của ArabicShuffled.")
    data["sex"] = data["sex"].map({"male": 0, "female": 1})
else:
    data = pd.read_csv(BytesIO(content), header=None)

if data.shape != EXPECTED_SHAPE or data.iloc[:, -1].isna().any():
    raise ValueError(f"Sai shape hoặc có nhãn thiếu: {data.shape}")

data = data.iloc[:150_000]
X = data.iloc[:, :-1].to_numpy(dtype=float)
y = data.iloc[:, -1].astype(str).to_numpy()
if not np.isfinite(X).all() or len(np.unique(y)) != EXPECTED_CLASSES:
    raise ValueError("Đặc trưng không hữu hạn hoặc số lớp không khớp.")

print(f"{DATASET}: {len(X):,} mẫu, {X.shape[1]} đặc trưng")
print(f"Huấn luyện ban đầu: {W:,} | Đánh giá: {len(X) - W:,}")

In [ ]:
# Cell 4 — Định nghĩa mô hình và thuật toán
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

RF = {"n_estimators": 100, "random_state": SEED, "n_jobs": -1}
XGB = {"n_estimators": 100, "random_state": SEED, "n_jobs": 2, "tree_method": "hist"}
PH = {"delta": 0.001}


def fit_predictor(kind, train_x, labels, seed=SEED):
    if kind == "RF":
        params = {**RF, "random_state": seed}
        return RandomForestClassifier(**params).fit(train_x, labels).predict
    encoder = LabelEncoder().fit(labels)
    if len(encoder.classes_) == 1:
        return DummyClassifier(strategy="most_frequent").fit(train_x, labels).predict
    params = {**XGB, "random_state": seed}
    model = XGBClassifier(**params).fit(train_x, encoder.transform(labels))

    def predict(features):
        return encoder.inverse_transform(model.predict(features))

    return predict


def fit_pair(start, stop):
    train_x = X[start:stop]
    teacher = fit_predictor(TEACHER, train_x, y[start:stop], seed=SEED)
    student = fit_predictor(STUDENT, train_x, teacher(train_x), seed=SEED + 100)
    return teacher, student


def run_studd():
    teacher, student = fit_pair(0, W)
    detector = PageHinkley(**PH)
    predictions, alarms = [], []
    start = W

    while start < len(X):
        stop = min(start + 512, len(X))
        teacher_predictions = teacher(X[start:stop])
        student_predictions = student(X[start:stop])

        for row, (t_pred, s_pred) in enumerate(zip(teacher_predictions, student_predictions), start):
            predictions.append(t_pred)
            detector.add_element(float(t_pred != s_pred))
            if detector.detected_change():
                alarms.append(row)
                start = row + 1
                if start < len(X):
                    teacher, student = fit_pair(start - W, start)
                    detector = PageHinkley(**PH)
                break
        else:
            start = stop

    return np.asarray(predictions), alarms

In [ ]:
# Cell 5 — Chạy BL-st và STUDD, xuất bảng so sánh
from pathlib import Path

from sklearn.metrics import accuracy_score, cohen_kappa_score

truth = y[W:]
baseline_pred = fit_predictor(TEACHER, X[:W], y[:W])(X[W:])
studd_pred, studd_alarms = run_studd()
runs = {"BL-st": (baseline_pred, []), "STUDD": (studd_pred, studd_alarms)}
summary = {}

for method, (predictions, alarms) in runs.items():
    assert len(predictions) == len(truth)
    kappa = float(cohen_kappa_score(truth, predictions))
    retrains = sum(row < len(X) - 1 for row in alarms)
    summary[method] = {
        "accuracy": float(accuracy_score(truth, predictions)),
        "kappa": kappa,
        "paper_kappa": PAPER_KAPPA[method],
        "kappa_relative_deviation_pct": 100 * abs(kappa - PAPER_KAPPA[method]) / abs(PAPER_KAPPA[method]),
        "drift_alarms": len(alarms),
        "retrains": int(retrains),
        "training_label_uses": W * (1 + int(retrains)),
    }

results = pd.DataFrame.from_dict(summary, orient="index").rename_axis("method")
output = Path(f"{SLUG}-seed{SEED}-summary.csv")
results.to_csv(output)
print(f"Đã lưu {output}")
display(results.round(4))

In [ ]:
# Cell 6 — Gửi kết quả. Đặt LOG_TO_WANDB=True ở cell 2 để bật.
def upload_runs():
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(
        host="https://api.wandb.ai",
        key=UserSecretsClient().get_secret("WANDB_API_KEY"),
        verify=True,
    )
    packages = ["numpy", "scipy", "pandas", "scikit-learn", "wandb"]
    if "XGB" in (TEACHER, STUDENT):
        packages.append("xgboost")

    for method, (predictions, alarms) in runs.items():
        config = {
            "dataset": DATASET, "source_url": DATA_URL, "source_sha256": EXPECTED_SHA256,
            "method": method, "seed": SEED, "initial_window": W,
            "stream_size": len(truth), "teacher": TEACHER,
            "teacher_params": RF if TEACHER == "RF" else XGB,
            "versions": {package: version(package) for package in packages},
        }
        if method == "STUDD":
            student_base = RF if STUDENT == "RF" else XGB
            config.update({
                "student": STUDENT,
                "student_seed": SEED + 100,
                "student_params": {**student_base, "random_state": SEED + 100},
                "detector_source": DETECTOR_SOURCE,
                "detector_source_url": DETECTOR_SOURCE_URL,
                "page_hinkley": {
                    "delta": PH["delta"], "min_instances": 30,
                    "threshold": 50, "alpha": 0.9999,
                },
                "retrain_window": W,
            })

        run_name = f"khoa-{SLUG}-{method.lower()}-paper-seed{SEED}"
        accuracy_200 = pd.Series(predictions == truth).rolling(200).mean()
        alarm_rows = set(alarms)
        with wandb.init(entity="lgkai-hcmus", project="STUDD", name=run_name,
                        group=f"khoa-{SLUG}-paper", config=config) as run:
            run.define_metric("stream_step")
            run.define_metric("accuracy_200", step_metric="stream_step")
            run.define_metric("alarm", step_metric="stream_step")
            for step, value in enumerate(accuracy_200, start=1):
                alarm = (W + step - 1) in alarm_rows
                if step % 25 == 0 or alarm or step == len(truth):
                    metrics = {"stream_step": step, "alarm": int(alarm)}
                    if pd.notna(value):
                        metrics["accuracy_200"] = float(value)
                    run.log(metrics)
            for metric, value in summary[method].items():
                run.summary[metric] = value
            run.summary["alarm_rows_zero_based"] = alarms
            url = run.url
        print(f"{method}: {url}")


if LOG_TO_WANDB:
    upload_runs()
else:
    print("W&B đang tắt. Đổi LOG_TO_WANDB ở cell 2, chạy lại cell 2 rồi chạy cell này.")

### Giấy phép detector

Lớp [PageHinkley của scikit-multiflow 0.5.3](https://github.com/scikit-multiflow/scikit-multiflow/blob/0.5.3/src/skmultiflow/drift_detection/page_hinkley.py) được nhúng bên dưới.

```text
BSD 3-Clause License

Copyright (c) 2017, scikit-multiflow
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.
* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

* Neither the name of the copyright holder nor the names of its
  contributors may be used to endorse or promote products derived from
  this software without specific prior written permission.
THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
```